# European Swaption Pricing and Risk

This notebook builds a transparent first model for a **European physically settled interest-rate swaption**. It is divided into short parts so that each idea can be discussed independently.

The notebook includes:

- deterministic discount and forward curves;
- the underlying forward-starting swap schedule;
- forward swap rate and swap annuity calculations;
- Black-76, shifted Black-76, and Bachelier pricing;
- payer/receiver parity;
- quote Greeks with explicit units;
- parallel and key-rate curve PV01;
- full-revaluation scenarios;
- numerical and theoretical validation.

> **Data status:** the curves and volatility below are illustrative. They allow the model to run and be validated without live market data. Production valuation requires calibrated curves, a swaption volatility cube, and complete market conventions.


## Part 1 — Scope and principal assumptions

The modeled contract gives its holder one right, at a single expiry date, to enter a fixed-for-floating interest-rate swap.

- A **payer swaption** gives the right to pay the fixed strike and receive floating. It generally benefits when swap rates rise.
- A **receiver swaption** gives the right to receive the fixed strike and pay floating. It generally benefits when swap rates fall.

The first implementation makes the following controlled simplifications:

1. expiry and payment times are year fractions, not calendar dates;
2. accrual periods are equal and determined by the fixed payment frequency;
3. curves are supplied as continuously compounded zero rates;
4. curve interpolation is linear in zero rates with flat-zero extrapolation;
5. the Black or Bachelier volatility is constant for this one expiry/tenor/strike point;
6. settlement is represented by the standard annuity-based physically settled European swaption formula;
7. collateral, calendars, stubs, business-day adjustments, and cash-settlement conventions are outside this first version.


In [ ]:
from dataclasses import dataclass, replace
import math
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = lambda x: f'{x:,.6f}'

## Part 2 — Contract, volatility quote, and curve objects

The contract stores legal/economic terms. The quote object stores the selected volatility convention. The zero curve maps a future time $t$ to a discount factor

$$P(0,t)=e^{-z(t)t},$$

where $z(t)$ is a continuously compounded zero rate.

Two curves are allowed:

- the **discount curve** present-values cash flows;
- the **projection curve** generates expected floating rates.

Keeping them separate makes the risk reports suitable for a multi-curve framework.


In [ ]:
@dataclass(frozen=True)
class EuropeanSwaption:
    expiry: float
    swap_tenor: float
    strike: float
    notional: float
    payer_receiver: Literal['payer', 'receiver']
    fixed_frequency: int = 2

    def validate(self):
        numeric = [self.expiry, self.swap_tenor, self.strike, self.notional]
        if not all(math.isfinite(float(x)) for x in numeric):
            raise ValueError('All contract inputs must be finite.')
        if self.expiry <= 0 or self.swap_tenor <= 0 or self.notional <= 0:
            raise ValueError('Expiry, swap tenor, and notional must be positive.')
        if self.payer_receiver.lower() not in {'payer', 'receiver'}:
            raise ValueError("payer_receiver must be 'payer' or 'receiver'.")
        if not isinstance(self.fixed_frequency, int) or self.fixed_frequency <= 0:
            raise ValueError('fixed_frequency must be a positive integer.')
        periods = self.swap_tenor * self.fixed_frequency
        if abs(periods - round(periods)) > 1e-10:
            raise ValueError('swap_tenor × fixed_frequency must be an integer in this simplified schedule.')


@dataclass(frozen=True)
class VolatilityQuote:
    model: Literal['black76', 'bachelier']
    volatility: float
    shift: float = 0.0

    def validate(self):
        if self.model.lower() not in {'black76', 'bachelier'}:
            raise ValueError("model must be 'black76' or 'bachelier'.")
        if not math.isfinite(self.volatility) or self.volatility < 0:
            raise ValueError('Volatility must be finite and non-negative.')
        if not math.isfinite(self.shift):
            raise ValueError('Shift must be finite.')
        if self.model.lower() == 'bachelier' and self.shift != 0.0:
            raise ValueError('The shift applies only to shifted Black-76, not Bachelier.')


@dataclass(frozen=True)
class ZeroCurve:
    name: str
    times: tuple
    rates: tuple

    def __post_init__(self):
        times = tuple(float(x) for x in self.times)
        rates = tuple(float(x) for x in self.rates)
        object.__setattr__(self, 'times', times)
        object.__setattr__(self, 'rates', rates)
        if len(times) != len(rates) or len(times) < 2:
            raise ValueError('Curve times and rates must have equal length of at least two.')
        if abs(times[0]) > 1e-14 or any(t < 0 for t in times):
            raise ValueError('The first curve time must be 0 and all times must be non-negative.')
        if any(b <= a for a, b in zip(times, times[1:])):
            raise ValueError('Curve times must be strictly increasing.')
        if not all(math.isfinite(x) for x in times + rates):
            raise ValueError('Curve inputs must be finite.')

    def zero_rate(self, time):
        x = np.asarray(time, dtype=float)
        if np.any(x < 0):
            raise ValueError('Discount times cannot be negative.')
        value = np.interp(x, self.times, self.rates, left=self.rates[0], right=self.rates[-1])
        return float(value) if x.ndim == 0 else value

    def discount(self, time):
        x = np.asarray(time, dtype=float)
        value = np.exp(-np.asarray(self.zero_rate(x)) * x)
        return float(value) if x.ndim == 0 else value

    def parallel_shift(self, basis_points):
        shift = float(basis_points) / 10_000.0
        return replace(self, rates=tuple(rate + shift for rate in self.rates))

    def bump_node(self, node_index, basis_points):
        if not 0 <= node_index < len(self.rates):
            raise IndexError('Curve node index is out of range.')
        bumped = list(self.rates)
        bumped[node_index] += float(basis_points) / 10_000.0
        return replace(self, rates=tuple(bumped))


@dataclass(frozen=True)
class SwaptionResult:
    present_value: float
    forward_swap_rate: float
    swap_annuity: float
    intrinsic_present_value: float

## Part 3 — Underlying swap schedule

If the option expires at $T_0$ and the underlying swap has payment dates $T_1,\ldots,T_n$, the fixed leg pays at those dates. In this first model, each fixed accrual is

$$\alpha_i=1/f,$$

where $f$ is the fixed payment frequency. The floating reset intervals run from $T_{i-1}$ to $T_i$.


In [ ]:
def swap_schedule(contract: EuropeanSwaption) -> pd.DataFrame:
    contract.validate()
    periods = int(round(contract.swap_tenor * contract.fixed_frequency))
    accrual = 1.0 / contract.fixed_frequency
    reset_times = contract.expiry + np.arange(periods, dtype=float) * accrual
    payment_times = contract.expiry + np.arange(1, periods + 1, dtype=float) * accrual
    return pd.DataFrame({
        'Period': np.arange(1, periods + 1),
        'Reset time': reset_times,
        'Payment time': payment_times,
        'Accrual fraction': np.full(periods, accrual),
    })


## Part 4 — Swap annuity and forward swap rate

The present value of one unit of fixed coupon is the swap annuity

$$A(0)=\sum_{i=1}^{n}\alpha_iP_d(0,T_i),$$

where $P_d$ comes from the discount curve. Projection-curve forward rates are

$$L_i(0)=\frac{P_f(0,T_{i-1})/P_f(0,T_i)-1}{\alpha_i}.$$

The floating-leg present value per unit notional is

$$PV_{float}=\sum_{i=1}^{n}P_d(0,T_i)\alpha_iL_i(0),$$

so the forward swap rate is

$$F_0=\frac{PV_{float}}{A(0)}.$$


In [ ]:
def forward_swap_metrics(
    contract: EuropeanSwaption,
    discount_curve: ZeroCurve,
    projection_curve: ZeroCurve,
):
    schedule = swap_schedule(contract)
    reset_times = schedule['Reset time'].to_numpy()
    payment_times = schedule['Payment time'].to_numpy()
    accruals = schedule['Accrual fraction'].to_numpy()

    discount_factors = discount_curve.discount(payment_times)
    projection_start = projection_curve.discount(reset_times)
    projection_end = projection_curve.discount(payment_times)
    forward_rates = (projection_start / projection_end - 1.0) / accruals

    annuity = float(np.sum(accruals * discount_factors))
    floating_leg_pv = float(np.sum(accruals * discount_factors * forward_rates))
    forward_swap_rate = floating_leg_pv / annuity

    detail = schedule.copy()
    detail['Discount factor'] = discount_factors
    detail['Projected forward rate'] = forward_rates
    detail['Fixed annuity contribution'] = accruals * discount_factors
    detail['Floating PV contribution'] = accruals * discount_factors * forward_rates
    return annuity, forward_swap_rate, detail


## Part 5 — Black-76 and Bachelier valuation

For shifted Black-76, define $X=F_0+s$ and $K_s=K+s$, where $s$ is the shift. For positive $X$ and $K_s$,

$$d_1=\frac{\ln(X/K_s)+\tfrac12\sigma_B^2T}{\sigma_B\sqrt{T}},\qquad d_2=d_1-\sigma_B\sqrt{T}.$$

The payer value is

$$V_{payer}=N A(0)[X\Phi(d_1)-K_s\Phi(d_2)].$$

Bachelier instead assumes a normally distributed forward swap rate. With normal volatility $\sigma_N$, $z=(F_0-K)/(\sigma_N\sqrt{T})$, and

$$V_{payer}=N A(0)[(F_0-K)\Phi(z)+\sigma_N\sqrt{T}\phi(z)].$$

Receiver values follow from the corresponding put formulas. The model quote convention must match the market volatility quote.


In [ ]:
def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def normal_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def price_from_forward(
    contract: EuropeanSwaption,
    quote: VolatilityQuote,
    annuity: float,
    forward_swap_rate: float,
) -> float:
    contract.validate()
    quote.validate()
    if annuity <= 0 or not math.isfinite(annuity):
        raise ValueError('Swap annuity must be finite and positive.')

    sign = 1.0 if contract.payer_receiver.lower() == 'payer' else -1.0
    intrinsic_rate = max(sign * (forward_swap_rate - contract.strike), 0.0)
    if quote.volatility == 0.0:
        return contract.notional * annuity * intrinsic_rate

    sqrt_t = math.sqrt(contract.expiry)
    if quote.model.lower() == 'black76':
        shifted_forward = forward_swap_rate + quote.shift
        shifted_strike = contract.strike + quote.shift
        if shifted_forward <= 0 or shifted_strike <= 0:
            raise ValueError('Shifted Black-76 requires forward + shift and strike + shift to be positive.')
        standard_deviation = quote.volatility * sqrt_t
        d1 = math.log(shifted_forward / shifted_strike) / standard_deviation + 0.5 * standard_deviation
        d2 = d1 - standard_deviation
        if sign > 0:
            rate_option_value = shifted_forward * normal_cdf(d1) - shifted_strike * normal_cdf(d2)
        else:
            rate_option_value = shifted_strike * normal_cdf(-d2) - shifted_forward * normal_cdf(-d1)
    else:
        standard_deviation = quote.volatility * sqrt_t
        z = (forward_swap_rate - contract.strike) / standard_deviation
        if sign > 0:
            rate_option_value = (
                (forward_swap_rate - contract.strike) * normal_cdf(z)
                + standard_deviation * normal_pdf(z)
            )
        else:
            rate_option_value = (
                (contract.strike - forward_swap_rate) * normal_cdf(-z)
                + standard_deviation * normal_pdf(z)
            )

    return contract.notional * annuity * rate_option_value


def price_swaption(
    contract: EuropeanSwaption,
    quote: VolatilityQuote,
    discount_curve: ZeroCurve,
    projection_curve: ZeroCurve,
) -> SwaptionResult:
    annuity, forward_swap_rate, _ = forward_swap_metrics(contract, discount_curve, projection_curve)
    present_value = price_from_forward(contract, quote, annuity, forward_swap_rate)
    sign = 1.0 if contract.payer_receiver.lower() == 'payer' else -1.0
    intrinsic_pv = contract.notional * annuity * max(
        sign * (forward_swap_rate - contract.strike), 0.0
    )
    return SwaptionResult(
        present_value=present_value,
        forward_swap_rate=forward_swap_rate,
        swap_annuity=annuity,
        intrinsic_present_value=intrinsic_pv,
    )

## Part 6 — Illustrative market and baseline valuation

The sample contract is a **1Y into 5Y payer swaption**: the option expires in one year and, if exercised, enters a five-year semiannual-pay swap. The notional is 10 million and the strike is 4.50%.

The example uses separate OIS discount and term projection curves. The 20% Black volatility is illustrative and is not calibrated to a market volatility cube.


In [ ]:
discount_curve = ZeroCurve(
    name='Illustrative OIS discount curve',
    times=(0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0),
    rates=(0.0350, 0.0360, 0.0370, 0.0390, 0.0400, 0.0415, 0.0420, 0.0425),
)
projection_curve = ZeroCurve(
    name='Illustrative term projection curve',
    times=(0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0),
    rates=(0.0380, 0.0390, 0.0405, 0.0420, 0.0430, 0.0440, 0.0445, 0.0450),
)
contract = EuropeanSwaption(
    expiry=1.0,
    swap_tenor=5.0,
    strike=0.0450,
    notional=10_000_000.0,
    payer_receiver='payer',
    fixed_frequency=2,
)
black_quote = VolatilityQuote(model='black76', volatility=0.20, shift=0.0)

schedule = swap_schedule(contract)
annuity, forward_swap_rate, cashflow_detail = forward_swap_metrics(
    contract, discount_curve, projection_curve
)
result = price_swaption(contract, black_quote, discount_curve, projection_curve)

baseline = pd.Series({
    'Present value': result.present_value,
    'Forward swap rate': result.forward_swap_rate,
    'Strike': contract.strike,
    'Swap annuity per unit notional': result.swap_annuity,
    'Intrinsic present value': result.intrinsic_present_value,
    'Black volatility': black_quote.volatility,
})
display(baseline.to_frame('Value'))
display(cashflow_detail)

## Part 7 — Payer/receiver parity

For otherwise identical European swaptions, payer-minus-receiver parity is

$$V_{payer}-V_{receiver}=N A(0)(F_0-K).$$

The right-hand side is the present value of entering the underlying forward swap as fixed-rate payer. This identity is an important implementation check and holds under both Black-76 and Bachelier.


In [ ]:
receiver_contract = replace(contract, payer_receiver='receiver')
payer_pv = price_swaption(contract, black_quote, discount_curve, projection_curve).present_value
receiver_pv = price_swaption(receiver_contract, black_quote, discount_curve, projection_curve).present_value
forward_swap_pv = contract.notional * annuity * (forward_swap_rate - contract.strike)

parity = pd.Series({
    'Payer swaption PV': payer_pv,
    'Receiver swaption PV': receiver_pv,
    'Payer PV - receiver PV': payer_pv - receiver_pv,
    'Underlying forward swap PV': forward_swap_pv,
    'Parity error': payer_pv - receiver_pv - forward_swap_pv,
})
display(parity.to_frame('Value'))

## Part 8 — Quote Greeks

Quote Greeks hold the swap annuity fixed and change the quoted forward rate or volatility. They are not the same as curve risk because curve movements can change both the forward swap rate and annuity.

The report uses practical units:

- forward Delta: PV change for a +1 bp forward swap-rate move;
- forward Gamma: second derivative scaled to a one-basis-point squared unit, so the quadratic P&L term for an $m$ bp move is $\tfrac12\Gamma_{bp^2}m^2$;
- Black Vega: PV change for one Black volatility point, meaning an absolute volatility move of 0.01;
- normal Vega: PV change for one basis point of normal volatility, meaning an absolute normal-volatility move of 0.0001.


In [ ]:
def analytic_quote_risk(
    contract: EuropeanSwaption,
    quote: VolatilityQuote,
    annuity: float,
    forward_swap_rate: float,
) -> pd.DataFrame:
    if quote.volatility <= 0:
        raise ValueError('Analytic Greeks require positive volatility.')
    sqrt_t = math.sqrt(contract.expiry)
    sign = 1.0 if contract.payer_receiver.lower() == 'payer' else -1.0

    if quote.model.lower() == 'black76':
        shifted_forward = forward_swap_rate + quote.shift
        shifted_strike = contract.strike + quote.shift
        if shifted_forward <= 0 or shifted_strike <= 0:
            raise ValueError('Shifted Black-76 Greek inputs must be positive.')
        standard_deviation = quote.volatility * sqrt_t
        d1 = math.log(shifted_forward / shifted_strike) / standard_deviation + 0.5 * standard_deviation
        delta_rate = normal_cdf(d1) if sign > 0 else normal_cdf(d1) - 1.0
        gamma_rate = normal_pdf(d1) / (shifted_forward * standard_deviation)
        vega_rate = shifted_forward * normal_pdf(d1) * sqrt_t
        vega_scale = 0.01
        vega_unit = 'PV per 1 Black vol point (0.01)'
    else:
        standard_deviation = quote.volatility * sqrt_t
        z = (forward_swap_rate - contract.strike) / standard_deviation
        delta_rate = normal_cdf(z) if sign > 0 else normal_cdf(z) - 1.0
        gamma_rate = normal_pdf(z) / standard_deviation
        vega_rate = normal_pdf(z) * sqrt_t
        vega_scale = 0.0001
        vega_unit = 'PV per 1 bp normal vol (0.0001)'

    scale = contract.notional * annuity
    return pd.DataFrame([
        {
            'Risk measure': 'Forward Delta',
            'Value': scale * delta_rate * 0.0001,
            'Unit': 'PV per +1 bp forward-rate move',
        },
        {
            'Risk measure': 'Forward Gamma',
            'Value': scale * gamma_rate * 0.0001**2,
            'Unit': 'Second derivative scaled per bp squared',
        },
        {
            'Risk measure': 'Vega',
            'Value': scale * vega_rate * vega_scale,
            'Unit': vega_unit,
        },
    ]).set_index('Risk measure')


quote_risk = analytic_quote_risk(contract, black_quote, annuity, forward_swap_rate)
display(quote_risk)

## Part 9 — Parallel discount and projection curve PV01

Curve PV01 is calculated by full central bump-and-revalue. A signed PV01 is reported:

$$PV01=\frac{V(+1\text{ bp})-V(-1\text{ bp})}{2}.$$

This is approximately the PV change caused by a +1 bp curve shift. Discount and projection curves are shocked separately because they represent different risk factors. A third result shifts both curves together.


In [ ]:
def parallel_curve_pv01(contract, quote, discount_curve, projection_curve, bump_bp=1.0):
    if bump_bp <= 0:
        raise ValueError('bump_bp must be positive.')

    def pv(discount, projection):
        return price_swaption(contract, quote, discount, projection).present_value

    discount_up = pv(discount_curve.parallel_shift(bump_bp), projection_curve)
    discount_down = pv(discount_curve.parallel_shift(-bump_bp), projection_curve)
    projection_up = pv(discount_curve, projection_curve.parallel_shift(bump_bp))
    projection_down = pv(discount_curve, projection_curve.parallel_shift(-bump_bp))
    both_up = pv(discount_curve.parallel_shift(bump_bp), projection_curve.parallel_shift(bump_bp))
    both_down = pv(discount_curve.parallel_shift(-bump_bp), projection_curve.parallel_shift(-bump_bp))

    return pd.Series({
        'Discount curve PV01': (discount_up - discount_down) / (2.0 * bump_bp),
        'Projection curve PV01': (projection_up - projection_down) / (2.0 * bump_bp),
        'Both curves parallel PV01': (both_up - both_down) / (2.0 * bump_bp),
    })


parallel_pv01 = parallel_curve_pv01(
    contract, black_quote, discount_curve, projection_curve
)
display(parallel_pv01.to_frame('Signed PV change per +1 bp'))

## Part 10 — Key-rate PV01

A parallel PV01 hides where curve risk occurs. Key-rate PV01 bumps one zero-rate pillar at a time and lets the interpolation create a local triangular shock around that pillar.

The results depend on the selected curve pillars and interpolation method. They are therefore model-defined risk measures, not unique properties of the trade.


In [ ]:
def key_rate_pv01(contract, quote, discount_curve, projection_curve, bump_bp=1.0):
    if bump_bp <= 0:
        raise ValueError('bump_bp must be positive.')

    def pv(discount, projection):
        return price_swaption(contract, quote, discount, projection).present_value

    rows = []
    for index in range(1, len(discount_curve.times)):  # time-zero node has no discount effect
        up = pv(discount_curve.bump_node(index, bump_bp), projection_curve)
        down = pv(discount_curve.bump_node(index, -bump_bp), projection_curve)
        rows.append({
            'Curve': 'Discount',
            'Pillar (years)': discount_curve.times[index],
            'Key-rate PV01': (up - down) / (2.0 * bump_bp),
        })

    for index in range(1, len(projection_curve.times)):
        up = pv(discount_curve, projection_curve.bump_node(index, bump_bp))
        down = pv(discount_curve, projection_curve.bump_node(index, -bump_bp))
        rows.append({
            'Curve': 'Projection',
            'Pillar (years)': projection_curve.times[index],
            'Key-rate PV01': (up - down) / (2.0 * bump_bp),
        })

    return pd.DataFrame(rows).set_index(['Curve', 'Pillar (years)'])


key_rate_risk = key_rate_pv01(
    contract, black_quote, discount_curve, projection_curve
)
display(key_rate_risk)

key_rate_plot = key_rate_risk.reset_index()
for curve_name, group in key_rate_plot.groupby('Curve'):
    plt.plot(group['Pillar (years)'], group['Key-rate PV01'], marker='o', label=curve_name)
plt.axhline(0.0, color='black', lw=0.8)
plt.xlabel('Curve pillar (years)')
plt.ylabel('Signed PV change per +1 bp')
plt.title('Swaption key-rate PV01')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Part 11 — Full-revaluation scenarios

Greeks approximate small moves. Scenario analysis shocks curves and volatility together, then fully recalculates the annuity, forward swap rate, exercise payoff distribution, and present value.

For this Black-76 example, a volatility shift of `0.05` means five Black volatility points. A Bachelier scenario would instead use an absolute normal-volatility shift, such as `0.0010` for ten basis points of normal volatility.


In [ ]:
@dataclass(frozen=True)
class SwaptionScenario:
    name: str
    discount_shift_bp: float = 0.0
    projection_shift_bp: float = 0.0
    volatility_shift: float = 0.0


def swaption_scenario_report(contract, quote, discount_curve, projection_curve, scenarios):
    base_pv = price_swaption(contract, quote, discount_curve, projection_curve).present_value
    rows = []
    for scenario in scenarios:
        shocked_quote = replace(quote, volatility=quote.volatility + scenario.volatility_shift)
        shocked_discount = discount_curve.parallel_shift(scenario.discount_shift_bp)
        shocked_projection = projection_curve.parallel_shift(scenario.projection_shift_bp)
        shocked_result = price_swaption(
            contract, shocked_quote, shocked_discount, shocked_projection
        )
        rows.append({
            'Scenario': scenario.name,
            'Discount shift (bp)': scenario.discount_shift_bp,
            'Projection shift (bp)': scenario.projection_shift_bp,
            'Volatility shift': scenario.volatility_shift,
            'Forward swap rate': shocked_result.forward_swap_rate,
            'Swap annuity': shocked_result.swap_annuity,
            'Present value': shocked_result.present_value,
            'Full-revaluation P&L': shocked_result.present_value - base_pv,
        })
    return pd.DataFrame(rows).set_index('Scenario')


scenarios = [
    SwaptionScenario('Base'),
    SwaptionScenario('Both curves +100 bp', 100.0, 100.0, 0.0),
    SwaptionScenario('Both curves -100 bp', -100.0, -100.0, 0.0),
    SwaptionScenario('Black vol +5 points', 0.0, 0.0, 0.05),
    SwaptionScenario('Black vol -5 points', 0.0, 0.0, -0.05),
    SwaptionScenario('Payer adverse combined', -50.0, -100.0, -0.05),
    SwaptionScenario('Payer favorable combined', 50.0, 100.0, 0.05),
]

scenario_results = swaption_scenario_report(
    contract, black_quote, discount_curve, projection_curve, scenarios
)
display(scenario_results)

## Part 12 — Numerical and theoretical validation

The validation block checks:

1. payer/receiver parity under Black-76;
2. payer/receiver parity under Bachelier;
3. analytic quote Greeks against central finite differences;
4. non-negative option values;
5. convergence to intrinsic value when volatility is zero.

These tests detect many implementation errors, but production validation should also compare against an independent pricing library and actual market quotes.


In [ ]:
# Black payer/receiver parity.
assert abs(payer_pv - receiver_pv - forward_swap_pv) < 1e-6

# Bachelier payer/receiver parity using an illustrative 80 bp normal volatility.
normal_quote = VolatilityQuote(model='bachelier', volatility=0.0080)
normal_payer = price_swaption(contract, normal_quote, discount_curve, projection_curve).present_value
normal_receiver = price_swaption(receiver_contract, normal_quote, discount_curve, projection_curve).present_value
assert abs(normal_payer - normal_receiver - forward_swap_pv) < 1e-6

# Analytic Black quote Greeks versus central finite differences with annuity fixed.
analytic = analytic_quote_risk(contract, black_quote, annuity, forward_swap_rate)['Value']
forward_bump = 0.0001
pv_base = price_from_forward(contract, black_quote, annuity, forward_swap_rate)
pv_forward_up = price_from_forward(contract, black_quote, annuity, forward_swap_rate + forward_bump)
pv_forward_down = price_from_forward(contract, black_quote, annuity, forward_swap_rate - forward_bump)
numeric_delta_per_bp = (pv_forward_up - pv_forward_down) / 2.0
numeric_gamma_per_bp_squared = pv_forward_up - 2.0 * pv_base + pv_forward_down

vol_bump = 0.0001
pv_vol_up = price_from_forward(
    contract, replace(black_quote, volatility=black_quote.volatility + vol_bump), annuity, forward_swap_rate
)
pv_vol_down = price_from_forward(
    contract, replace(black_quote, volatility=black_quote.volatility - vol_bump), annuity, forward_swap_rate
)
numeric_vega_per_vol_point = (pv_vol_up - pv_vol_down) / (2.0 * vol_bump) * 0.01

assert abs(numeric_delta_per_bp - analytic['Forward Delta']) < 0.05
assert abs(numeric_gamma_per_bp_squared - analytic['Forward Gamma']) < 0.05
assert abs(numeric_vega_per_vol_point - analytic['Vega']) < 0.05

# Non-negativity and zero-volatility limit.
assert payer_pv >= 0.0 and receiver_pv >= 0.0
zero_vol_quote = replace(black_quote, volatility=0.0)
zero_vol_pv = price_swaption(contract, zero_vol_quote, discount_curve, projection_curve).present_value
assert abs(zero_vol_pv - result.intrinsic_present_value) < 1e-10

validation = pd.Series({
    'Black parity error': payer_pv - receiver_pv - forward_swap_pv,
    'Bachelier parity error': normal_payer - normal_receiver - forward_swap_pv,
    'Forward Delta analytic-minus-numeric': analytic['Forward Delta'] - numeric_delta_per_bp,
    'Forward Gamma analytic-minus-numeric': analytic['Forward Gamma'] - numeric_gamma_per_bp_squared,
    'Vega analytic-minus-numeric': analytic['Vega'] - numeric_vega_per_vol_point,
    'Zero-volatility limit error': zero_vol_pv - result.intrinsic_present_value,
})
display(validation.to_frame('Error'))
print('All core swaption validation checks passed.')

## Part 13 — Market data needed for production use

The illustrative inputs are enough for learning and model testing. A market-calibrated valuation requires:

1. valuation date and reporting currency;
2. swaption expiry, underlying swap tenor, strike, payer/receiver direction, notional, and settlement type;
3. fixed- and floating-leg indices, frequencies, day-count conventions, calendars, business-day rules, and any stubs;
4. collateral/discounting curve with instrument definitions and bootstrapping conventions;
5. the appropriate forward projection curve for the floating index;
6. swaption volatility cube by option expiry, swap tenor, and strike or moneyness;
7. quote convention: Black, shifted Black, or normal volatility, including the shift where relevant;
8. market premiums for calibration and independent benchmark comparison;
9. cash-settlement methodology if the contract is not physically settled.


## Part 14 — Controlled next extensions

After the European model is understood and independently validated, the recommended development order is:

1. date-based schedules, calendars, day counts, and stubs;
2. curve bootstrapping from deposits, futures/FRAs, OIS, and swaps;
3. implied-volatility inversion and volatility-cube interpolation;
4. SABR smile calibration and strike risk;
5. Theta, carry, roll-down, and daily P&L explain using a valuation-date engine;
6. cash-settled swaption conventions;
7. Bermudan swaptions using a calibrated short-rate lattice, PDE, or Monte Carlo model;
8. independent benchmarking, limits, backtesting, and model governance.

A Bermudan swaption will reintroduce the exercise-versus-continuation decision from the American-option project, but the state variable will be the interest-rate term structure rather than one equity price.
